# Real-data leave-one-out (d = 10 / 20 / 50) -- statefate

For each held interior day `t*`, fit each method's prediction of the held marginal from the OTHER
days, save `pred_*.npy`, and score MMD / EMD / W2 against the true held-out cells. Methods in this
notebook: **ours** (the UOTReg barycenter estimated directly at `t*`), the balanced-OT variant,
naive-midpoint, carry-forward, and MMFM. TIGON and MioFlow run in their own environments; their
committed predictions ship under `results/realdata/` and `loo_figs` stitches all methods together.
Run `DIM` = 10 / 20 / 50 for the d-scaling table.

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO = P.REPO
import uotreg as U
from uotreg import baselines as B
from uotreg.metrics import mmd_rbf, emd, w2
from uotreg import datasets


def silverman(times):
    """Silverman-rule bandwidth over the time grid (matches realdata_dist_est_dims_statefate's h_bench)."""
    X = np.asarray(times, float)
    return float(1.06 * np.std(X) * len(X) ** (-1 / 5))

## Parameters
`DIM` = 10 / 20 / 50 (d=50 needs the .h5ad). `HELD_DAYS` = the interior days (keep them identical
across the LOO files so the rows are comparable). `EST` = the estimator config of record for
statefate; `READOUT_START` picks where MMFM integrates from.

In [ ]:
DATASET = "statefate"
DIM     = 20                       # 10 | 20 | 50
DEVICE  = "cpu"
# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE   = 1  # 1 = tiny/fast plumbing check; 0 = the paper's numbers
# 1 = write results/figures to `new_results/`; 0 = keep everything in memory.
# The shipped `results/` tree is never modified either way.
SAVE = 0

HELD_DAYS     = [4.0]              # statefate interior day (times 2, 4, 6)
N_PRED        = 400 if SMOKE else 1500   # points generated per prediction
BENCH_REPEATS = 3 if SMOKE else 50       # metric CI repeats
BENCH_N       = 400 if SMOKE else 1000   # points/side per metric draw
READOUT_START = "ends"             # MMFM integration start: "ends" (global) | "neighbors"

# Initialization (mirrors realdata_dist_est_dims_statefate): default "gaussian" (broad blob, scale 10).
# Set "vae_nf" for the data-aware flow-VAE init (encoder + this-generator-as-decoder + planar flows on
# the POOLED kept-day cells) that captures off-centre modes a Gaussian blob misses. `INIT_STRATEGY` is
# ignored at d=20 (the saved 256-wide ini is used instead). We don't need vae_nf, but it's wired.
INIT_STRATEGY = "gaussian"                          # "gaussian" | "vae_nf" | "data_gaussian" | "flow"
VAE_EPOCHS, VAE_FLOW_LENGTH, VAE_COEF = 5, 16, 5.0  # vae_nf hyperparameters (match the reference)

# EST mirrors `reproduce/realdata/realdata_dist_est_dims_statefate.py` EXACTLY: RAW PC space
# (std_mode="none"), KL one-sided UOT, cost sqeuclidean (estimate's default), tau=1, arch gen 256/4 +
# map/pot 196/5 (gen_dropout=dropout=0.05, batchnorm=False), train schedule d/t/g=50/10/50, batch
# 64/128, lr 3e-4/3e-4/1e-4, wd 1e-10/1e-8, budget (outer TRAIN_OUTER_BENCH)=85 (the real leave-one-out
# budget -- the estimate needs many rounds; NOT the small 40 default), gaussian init scale 10 iters
# 10000 (or the saved d=20 ini). `h` is NOT fixed here: it is the Silverman bandwidth on the KEPT days
# (statefate uses h_bench=None -> Silverman in the reference), computed per held day in `_est` (~1.85 for [2,6]).
EST = dict(tau=1.0, std_mode="none", divergence="kl",
           gen_hidden=(64 if SMOKE else 256), gen_layers=4, map_hidden=(64 if SMOKE else 196), map_layers=5,
           d_iters=(15 if SMOKE else 50), t_iters=(5 if SMOKE else 10), g_iters=(15 if SMOKE else 50),
           batch_size=(32 if SMOKE else 64), batch_size_g=(64 if SMOKE else 128),
           budget=(6 if SMOKE else 85), init=INIT_STRATEGY,
           init_iters=(500 if SMOKE else 10000), gaussian_scale=10.0,
           vae_epochs=VAE_EPOCHS, vae_flow_length=VAE_FLOW_LENGTH, vae_coef=VAE_COEF,
           device=DEVICE)
INI_PATH = P.aux_path("data/ini/G_statefate20_256_Dayall_ini.pth")
USE_INI  = (DIM == 20) and (not SMOKE) and os.path.exists(INI_PATH)  # ini is 256-wide -> full scale only
MMFM = dict(hidden=(64 if SMOKE else 256), iters=(300 if SMOKE else 3000),
            n_tuples=(120 if SMOKE else 200))
_here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
# this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
for _d in (_here, os.path.abspath(os.path.join(_here, os.pardir, os.pardir, "tools"))):
    if _d not in sys.path:
        sys.path.insert(0, _d)
import _loo_paths as _P          # per-method output folders; imports nothing but `os`
# SMOKE predictions go to their own tree: they are 400-point budget-6 clouds that would
# otherwise overwrite the shipped benchmark predictions loo_figs reads.
RESULTS = P.results("realdata_smoke" if SMOKE else "realdata", write=True)
if SAVE:
    os.makedirs(RESULTS, exist_ok=True)

## Data
Per-time PC clouds in time order. `SMOKE` subsamples the cells for a fast pass; held days must be
interior (a trajectory read-out needs neighbors on both sides).

In [ ]:
ds = datasets.load_statefate(P.DATA_DIR, d=DIM)
timepoints = ds.timepoints.tolist()
arrays = [np.asarray(a, np.float32) for a in ds.arrays]
if SMOKE:
    _r = np.random.default_rng(0)
    arrays = [a[_r.choice(len(a), min(1200, len(a)), replace=False)] for a in arrays]
print(f"{DATASET} d={DIM}: times={timepoints}  cells/snapshot={[len(a) for a in arrays]}")
for h in HELD_DAYS:
    j = timepoints.index(h)
    assert 0 < j < len(timepoints) - 1, f"held day {h} is not interior"

## Method read-outs (fit on the kept days, predict the held marginal `t*`)
* **ours** — `U.estimate` at `t*` from the kept days (the Fréchet-UOT barycenter; our LOO readout).
* **MMFM** — one field fit on the kept days, integrated to `t*` from an early kept day (forward) and
  a late kept day (backward), like the trajectory-baselines file.
* **naive-midpoint / carry-forward** — the reference baselines.

In [ ]:
def _fwd_bwd_starts(kept_times, tstar):
    if READOUT_START == "neighbors":
        fi = max(i for i, t in enumerate(kept_times) if t < tstar)
        bi = min(i for i, t in enumerate(kept_times) if t > tstar)
    else:
        fi, bi = 0, len(kept_times) - 1
    return fi, bi


def _est(kept_arrays, kept_times, tstar, relaxation):
    # h = Silverman on the KEPT days (matches realdata_dist_est_dims_statefate: h_bench=None -> Silverman)
    kw = dict(EST, relaxation=relaxation, h=silverman(kept_times))
    if USE_INI:
        kw["pretrained_generator"] = INI_PATH
    return U.estimate(kept_arrays, kept_times, query_time=tstar, dim=DIM, n_gen=N_PRED, seed=0, **kw)


def mmfm_readouts(kept_arrays, kept_times, tstar):
    field = B.mmfm_fit(kept_arrays, kept_times, dim=DIM, hidden=MMFM["hidden"], iters=MMFM["iters"],
                       n_tuples=MMFM["n_tuples"], sigma=0.05, device=DEVICE, seed=0)
    fi, bi = _fwd_bwd_starts(kept_times, tstar)
    fwd = B.rk4_sample(field, kept_arrays[fi][:N_PRED], [kept_times[fi], tstar], n_per=20, device=DEVICE)[-1]
    bwd = B.rk4_sample(field, kept_arrays[bi][:N_PRED], [kept_times[bi], tstar], n_per=20, device=DEVICE)[-1]
    return fwd, bwd


def cell_sampler(cells):
    cells = np.asarray(cells)
    return lambda n: cells[np.random.default_rng().integers(0, len(cells), n)]

## Leave-one-out loop — predict, SAVE, SCORE
For each held day: build every method's prediction, save `pred_<method>.npy`, and score MMD/EMD/W2
vs the true held-out cells over `BENCH_REPEATS` draws (lower = better).

In [ ]:
metric_fns = {"MMD": mmd_rbf, "EMD": emd, "W2": w2}
preds_all = {}
for HELD in HELD_DAYS:
    j = timepoints.index(HELD)
    kept_idx = [i for i in range(len(timepoints)) if i != j]
    kept_times = [timepoints[i] for i in kept_idx]
    kept_arrays = [arrays[i] for i in kept_idx]
    t0 = time.time()
    fwd, bwd = mmfm_readouts(kept_arrays, kept_times, HELD)
    b, a = arrays[j - 1], arrays[j + 1]; rng = np.random.default_rng(0)
    preds = {
        "UOTReg":         _est(kept_arrays, kept_times, HELD, "one-sided"),
        "OT":             _est(kept_arrays, kept_times, HELD, "balanced"),
        "MMFM fwd":       fwd,
        "MMFM bwd":       bwd,
        "Naive1 (mid)":   0.5 * (b[rng.integers(0, len(b), N_PRED)] + a[rng.integers(0, len(a), N_PRED)]),
        "Naive2 (carry)": b[rng.integers(0, len(b), N_PRED)],
    }
    preds_all[HELD] = preds
    for nm, g in preds.items():
        # distinct keys for the two MMFM read-outs so they never overwrite each other
        key = {"MMFM fwd": "MMFMloo_fwd", "MMFM bwd": "MMFMloo_bwd"}.get(nm, nm.split()[0])
        if SAVE:
            np.save(os.path.join(_P.out_dir(RESULTS, key),
                             f"pred_{DATASET}{DIM}_Day{HELD}_{key}.npy"), np.asarray(g))
    truth = arrays[j]; st = cell_sampler(truth)
    res = {m: {k: [] for k in metric_fns} for m in preds}
    for _ in range(BENCH_REPEATS):
        ref = st(BENCH_N)
        for m, g in preds.items():
            p = cell_sampler(g)(BENCH_N)
            for k, fn in metric_fns.items():
                res[m][k].append(fn(p, ref))
    print(f"\n[{DATASET} d={DIM}] held Day{HELD} ({BENCH_REPEATS} reps, N={BENCH_N}; lower=better; "
          f"fit+readout {time.time()-t0:.0f}s):")
    print(f"  {'method':16s}" + "".join(f"{k:>15s}" for k in metric_fns))
    for m in preds:
        print(f"  {m:16s}" + "".join(f"{np.mean(res[m][k]):7.3f}+/-{np.std(res[m][k]):<5.3f}" for k in metric_fns))

## Visualize predictions vs true held-out cells (dims 0,1)
One row per held day: true cells (grey) with each method's predicted cloud (purple).

In [ ]:
for HELD in HELD_DAYS:
    j = timepoints.index(HELD); truth = arrays[j]
    panels = [("true held-out", truth)] + list(preds_all[HELD].items())
    fig, axes = plt.subplots(1, len(panels), figsize=(2.7 * len(panels), 3.0), dpi=110, sharex=True, sharey=True)
    for ax, (nm, g) in zip(np.atleast_1d(axes), panels):
        g = np.asarray(g)
        ax.scatter(truth[:, 0], truth[:, 1], s=5, c="0.75", alpha=0.3)
        if nm != "true held-out":
            ax.scatter(g[:, 0], g[:, 1], s=6, c="tab:purple", alpha=0.5)
        ax.set_title(nm, fontsize=8)
    fig.suptitle(f"[{DATASET} d={DIM}] Day{HELD}: predictions vs true (dims 0,1)")
    fig.tight_layout(); plt.show()